In [ ]:
# CELL 1. Install everything BEFORE starting the server.
import os
import sys
import subprocess
import time
import json
import urllib.request
from pathlib import Path

def healthy():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=3) as r:
            return r.status == 200
    except OSError:
        return False

if healthy():
    print("СЕРВЕР ГОТОВ. Можно запускать вторую ячейку.")
else:
    result = subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "vllm==0.33.0", "openai", "mlflow==3.18.0", "matplotlib",
    ], capture_output=True, text=True)
    if result.returncode:
        print(result.stdout[-3000:])
        print(result.stderr[-10000:])
        raise RuntimeError("Установка остановилась; причина выше.")
    print("Библиотеки установлены.", flush=True)
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.free",
                          "--format=csv,noheader"], capture_output=True, text=True)
    print(gpu.stdout)
    if gpu.returncode or "T4" not in gpu.stdout:
        raise RuntimeError("В настройках Colab нужно выбрать T4 GPU.")

    command = """vllm serve Qwen/Qwen3-4B-Instruct-2507
    --dtype half --max-model-len 32768
    --gpu-memory-utilization 0.90 --max-num-seqs 32
    --max-num-batched-tokens 2048 --enforce-eager
    --enable-prefix-caching --enable-auto-tool-choice
    --tool-call-parser hermes --host 127.0.0.1 --port 8000""".split()
    log_path = Path("/content/vllm.log")
    if log_path.exists():
        previous = Path(f"/content/vllm_previous_{time.time_ns()}.log")
        previous.write_bytes(log_path.read_bytes())
    Path("/content/server_command.txt").write_text(" ".join(command))
    environment = dict(os.environ, PYTHONUNBUFFERED="1",
                       VLLM_WORKER_MULTIPROC_METHOD="spawn")
    with log_path.open("w") as log:
        server = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                                  env=environment, start_new_session=True)
    print("Загружается модель. Первое скачивание может занять несколько минут.", flush=True)
    for attempt in range(180):
        if server.poll() is not None:
            print(log_path.read_text(errors="replace")[-12000:])
            raise RuntimeError("Сервер остановился. Ошибка напечатана выше; вторую ячейку не запускай.")
        if healthy():
            print("СЕРВЕР ГОТОВ. Можно запускать вторую ячейку.")
            break
        if attempt and attempt % 12 == 0:
            print(f"Загрузка продолжается: {attempt * 5} секунд.", flush=True)
        time.sleep(5)
    else:
        print(log_path.read_text(errors="replace")[-12000:])
        raise RuntimeError("Сервер ещё не готов. Вторую ячейку пока не запускай.")


In [ ]:
# CELL 2. Data, checks, three experiments, plots, MLflow and backup.
import os
import sys
import subprocess
import urllib.request
from pathlib import Path
from google.colab import files

try:
    with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5) as r:
        if r.status != 200:
            raise RuntimeError("Сервер не готов")
except OSError as error:
    raise RuntimeError("Сначала дождись СЕРВЕР ГОТОВ в первой ячейке.") from error

project = Path("/content/lr1_clean")

def execute(command, cwd=None):
    process = subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        env=dict(os.environ, PYTHONUNBUFFERED="1"),
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError("Шаг остановлен. Причина напечатана выше.")

if not project.exists():
    execute(["git", "clone",
             "https://github.com/marchukmarina-creator/lr1-llm-inference.git",
             str(project)])
os.chdir(project)

run_id = "lr1_final"
has_results = (project / "runs" / run_id / "results.jsonl").exists()
if not has_results:
    # Explicit return type avoids Transformers 5 returning a dict instead of token IDs.
    source = Path("lr1.py")
    code = source.read_text(encoding="utf-8")
    old = 'tokenize=True, add_generation_prompt=True))'
    new = 'tokenize=True, add_generation_prompt=True, return_dict=False))'
    if old in code:
        source.write_text(code.replace(old, new), encoding="utf-8")
    elif new not in code:
        raise RuntimeError("Код на GitHub изменился: нужно проверить функцию подготовки запросов.")

    execute([sys.executable, "-m", "unittest", "-v", "test_lr1.py"])
    execute([sys.executable, "lr1.py", "prepare"])
    execute([sys.executable, "lr1.py", "check"])
    execute(["git", "add", "-A"])
    if subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode == 1:
        execute(["git", "-c", "user.name=LR1 preparation",
                 "-c", "user.email=lr1-preparation@example.invalid",
                 "commit", "-m",
                 "Fix tokenizer compatibility; record server, data and checks before experiments"])

print("\nНАЧИНАЮТСЯ ЭКСПЕРИМЕНТЫ", flush=True)
execute([sys.executable, "lr1.py", "run", "--experiment", "all", "--run-id", run_id])
execute([sys.executable, "lr1.py", "analyze", "--run-id", run_id])
execute([sys.executable, "lr1.py", "pack", "--run-id", run_id])
print("\nЗАПУСК ЗАВЕРШЁН. Таблицы, графики и проект отчёта сохранены.", flush=True)
files.download(str(project / f"lr1_{run_id}_artifacts.zip"))
